# Case Study 7.1 - Industry Vertical Labelling

In the first case study of chapter 7 we look at using a generative language model API to perform labelling of a dataset of media stories for the purpose of determining which industry vertical they are relevant to. We build on the ideas in chapter 6 using zero and few shot classification.

## 01 - Data Understanding

You will need to first download the 2022 NAICS dataset from the [U.S. government NAICS website](https://www.census.gov/naics/)

We grabbed the files '2022_NAICS_Descriptions.xlsx' and saved it in the data directory.


In [ ]:
import pandas as pd
from collections import defaultdict

# Load the Excel file
file_path = 'data/2022_NAICS_Descriptions.xlsx'
df = pd.read_excel(file_path, dtype={'Code': str})

# Clean up any rows with missing codes
df = df.dropna(subset=['Code'])

# Filter only valid codes (2-6 digits and numeric)
df = df[df['Code'].str.match(r'^\d{2,6}$')]

In [ ]:
# Get all root codes (2-digit)
root_nodes = df[df['Code'].str.len() == 2]
roots = {}
for _, root_row in root_nodes.iterrows():
    root_code = root_row['Code']
    title = root_row['Title'][:-1]
    if len(title)>27:
       title = title[:24] + "..."
    roots[root_code] = title


In [ ]:
roots["31"] = "Manufacturing: Foods, beverages and fabrics"
roots["32"] = "Manufacturing: Wood, chemicals, petroleum and plastics"
roots["33"] = "Manufacturing: Metals, Refining, Processing, Equipment, Vehicles, Electronics"
roots["44"] = "Retail Trade - Vehicles, Homewares, Foods, Beverages"
roots["45"] = "Retail Trade - Pharmaceuticals, Apparel, Lifestyle"
roots["48"] = "Transportation and Support Services"
roots["49"] = "Warehousing, Delivery and Logistics"

In [ ]:
# Create a dictionary to store counts
hierarchy_counts = defaultdict(lambda: {3: 0, 4: 0, 5: 0, 6: 0})

# Iterate through all entries
for _, row in df.iterrows():
    code = row['Code']
    if len(code) == 2:
        continue  # Skip root nodes themselves
    root_code = code[:2]  # First 2 digits determine the root node
    hierarchy_counts[root_code][len(code)] += 1


In [ ]:
results = pd.DataFrame(columns=['Code', 'Title', 'Lvl3', 'Lvl4', 'Lvl5', 'Lvl6'])

keys = list(roots.keys())
keys.sort()
for root_code in keys:
    title = roots[root_code]
    counts = hierarchy_counts.get(root_code, {3: 0, 4: 0, 5: 0, 6: 0})
    record = {"Code":root_code, "Title":title, 'Lvl3':counts[3], 'Lvl4':counts[4], 'Lvl5':counts[5], 'Lvl6':counts[6]}
    results = pd.concat([results,pd.DataFrame([record])], ignore_index=True)

print(results.to_markdown(index=False))

## 02 - Data Preparation

We next want to process the data so that we have reference files with the descriptions of categories at a given level in the hierarchy.

You will not that in the analysis above we needed to add some top level categories because we discovered inconsistencies in how the NAICS categories were captured in the spreadsheets. In the book we describe some edits we made to that raw file so that there was a consistent hierarchy we could analyse. After making those edits we saved the file as data/2022_NAICS_Descriptions_modified.xlsx

In [ ]:
import json
import pandas as pd

In [ ]:
# Load the Excel file
# We modified the original to add entries for the 3 manufacturing categories
file_path = 'data/2022_NAICS_Descriptions_modified.xlsx'
df = pd.read_excel(file_path, dtype={'Code': str})

In [ ]:
def clean_text(txt):
    txt = str(txt)
    if txt[-1:]=="T":
        txt =  txt[:-1]
    return txt

df['Title'] = df['Title'].apply(clean_text)

In [ ]:
# Get all root codes (2-digit)
root = df[df['Code'].str.len() == 2]

root_dict = pd.Series(root['Title'].values, index=root['Code']).to_dict()

with open("data/2022_NAICS_Roots.json", "w") as f:
    json.dump(root_dict, f)

In [ ]:
lvl3 = df[df['Code'].str.len() == 3]

# Now get all categories below each of the root levels and save into a reference file
for _, root_row in root.iterrows():
    root_code = root_row['Code']
    temp = lvl3[lvl3['Code'].str[0:2] == root_code]
    print(f"For {root_code} we have {len(temp)}")
    temp_dict = pd.Series(temp['Title'].values, index=temp['Code']).to_dict()
    file_name = "data/2022_NAICS_"+root_code+".json"
    with open(file_name, "w") as f:
        json.dump(temp_dict, f)

## 03 - Structured Outputs

When we categorise content we want our language models to return structured outputs. 
Here is how that is defined.

In [ ]:
import json
from enum import Enum
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
with open("data/2022_NAICS_Roots.json", 'r') as f:
    categories = json.load(f)

categories["00"] = "Unknown industry or content not related to any industry"

In [ ]:
CategoryEnum = Enum('CategoryEnum', categories)

# Define a Pydantic model using the dynamically created Enum
class NAICS_Category(BaseModel):
    category: CategoryEnum = Field(description="The NAICS industry category.")
    reason: str = Field(description="An explanation for the assigned category.")

parser = JsonOutputParser(pydantic_object=NAICS_Category)

## 04 - Language Model Classifier

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
   template = """
     Your job is to categorise a news article to determine which, if any,
     of the NAICS industry categories is most relevant to for investors.
     You are not categorising the content itself, but whether the news
     story contains information that will be pertinent to investors in
     a specific sector of the economy. Return the name of the industry
     category from the provided list of options.

     Here is data for you to process:
     ---
     article: {article}
     ---
     {format_instructions}
     """,
   input_variables=["article"],
   partial_variables={"format_instructions":parser.get_format_instructions()}
)

from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
chain = prompt | model | parser

In [ ]:
lookup = {v: k for k, v in categories.items()}

We create a silly example article to test the system:

In [ ]:
article = """
Last weekend residents of inner city Brisbane Australia were woken by the sound of the XXXX brewery exploding. 
Beer was sprayed across the city and resulted in the death of local wildlife
"""

In [ ]:
response = chain.invoke({"article":article})
code = lookup[response['category']]
print(f"Categorised as {code} : {response['category']} ")

## 05 - Hierarchical Categorization

We next need a process that will dig deeper into the hierarchy based on the initial classification.

To do this we create a new function that will return the appropriate sub-class categorization model depending on what the root category was.

In [ ]:
def get_chain(code, model):
   file_name = "data/2022_NAICS_"+code+".json"
   with open(file_name, 'r') as f:
      subcats = json.load(f)

   SubCatEnum = Enum('SubCategoryEnum', subcats)
   class SubCategory(BaseModel):
      subcategory: SubCatEnum = Field(description="The NAICS industry sub category.")
      reason: str = Field(description="An explanation for the assigned category.")

   subparser = JsonOutputParser(pydantic_object=SubCategory)

   subprompt = PromptTemplate(
      template = """
        Your job is to identify the most appropriate sub-category from the
        NAICS industry categories for the news article.
        You are not categorising the content itself, but whether the news
        story contains information that will be pertinent to investors in
        a specific sector of the economy. Return the name of the industry
        sub-category from the provided list of options.

        Here is data for you to process:
        ---
        article: {article}
        ---
        {format_instructions}
        """,
      input_variables=["article"],
      partial_variables={"format_instructions":subparser.get_format_instructions()}
   )
   subchain = subprompt | model | subparser
   return subchain


> Test the new function by applying it to our example

In [ ]:
subchain = get_chain(code, model)

response = subchain.invoke({"article":article})
print(response)

print(f"Categorised as {response['subcategory']} ")


## 06 - Post-Processing

In [ ]:

from langchain_core.runnables import Runnable, RunnableConfig
from typing import Any, Dict

class RetrieveCode(Runnable):
    def __init__(self, lookup, in_field, out_field, default="?"):
        self.lookup = lookup
        self.in_field = in_field
        self.out_field = out_field
        self.default = default

    def invoke(
        self,
        input: Any,
        config: RunnableConfig = None,
        **kwargs: Any,
    ) -> Any:
        # Implement the custom logic here
        if input[self.in_field] in self.lookup:
            input[self.out_field] = self.lookup[ input[self.in_field] ]
        else:
            input[self.out_field] = self.default
        return input



In [ ]:
def get_chain(code, model):
   file_name = "data/2022_NAICS_"+code+".json"
   with open(file_name, 'r') as f:
      subcats = json.load(f)

   SubCatEnum = Enum('SubCategoryEnum', subcats)
   class SubCategory(BaseModel):
      subcategory: SubCatEnum = Field(description="The NAICS industry sub category.")
      reason: str = Field(description="An explanation for the assigned category.")

   subparser = JsonOutputParser(pydantic_object=SubCategory)

   subprompt = PromptTemplate(
      template = """
        Your job is to identify the most appropriate sub-category from the
        NAICS industry categories for the news article.
        You are not categorising the content itself, but whether the news
        story contains information that will be pertinent to investors in
        a specific sector of the economy. Return the name of the industry
        sub-category from the provided list of options.

        Here is data for you to process:
        ---
        article: {article}
        ---
        {format_instructions}
        """,
      input_variables=["article"],
      partial_variables={"format_instructions":subparser.get_format_instructions()}
   )

   sublookup = {v: k for k, v in subcats.items()}
   post_process = RetrieveCode(sublookup, "subcategory", "code")

   subchain = subprompt | model | subparser | post_process
   return subchain

In [ ]:
subchain = get_chain(code, model)

response = subchain.invoke({"article":"Last weekend residents of inner city Brisbane Australia were woken by the sound of the XXXX brewery exploding. Beer was sprayed across the city and resulted in the death of local wildlife."})

print(f"Categorised as {response['code']} : {response['subcategory']} ")

## 08 - Complete Process

In [ ]:
import functools

@functools.lru_cache(maxsize=None)
def get_root_chain():
   with open("data/2022_NAICS_Roots.json", 'r') as f:
       categories = json.load(f)
   categories["00"] = "Unknown industry or content not related to any industry"
   CategoryEnum = Enum('CategoryEnum', categories)
   class NAICS_Category(BaseModel):
       category: CategoryEnum = Field(description="The NAICS industry category.")
       reason: str = Field(description="An explanation for the assigned category.")

   parser = JsonOutputParser(pydantic_object=NAICS_Category)
   prompt = PromptTemplate(
      template = """
        Your job is to categorise a news article to determine which, if any,
        of the NAICS industry categories it is most relevant to for investors.
        You are not categorising the content itself, but whether the news
        story contains information that will be pertinent to investors in
        a specific sector of the economy. Return the name of the industry
        category from the provided list of options.

        Here is data for you to process:
        ---
        article: {article}
        ---
        {format_instructions}
        """,
      input_variables=["article"],
      partial_variables={"format_instructions":parser.get_format_instructions()}
   )
   lookup = {v: k for k, v in categories.items()}
   post_process = RetrieveCode(lookup, "category", "code")
   chain = prompt | model | parser | post_process
   return chain

In [ ]:
@functools.lru_cache(maxsize=None)
def get_chain(code):
   file_name = "data/2022_NAICS_"+code+".json"
   with open(file_name, 'r') as f:
      subcats = json.load(f)

   SubCatEnum = Enum('SubCategoryEnum', subcats)
   class SubCategory(BaseModel):
      subcategory: SubCatEnum = Field(description="The NAICS industry sub category.")
      reason: str = Field(description="An explanation for the assigned category.")

   subparser = JsonOutputParser(pydantic_object=SubCategory)

   subprompt = PromptTemplate(
      template = """
        Your job is to identify the most appropriate sub-category from the
        NAICS industry categories for the news article.
        You are not categorising the content itself, but whether the news
        story contains information that will be pertinent to investors in
        a specific sector of the economy. Return the name of the industry
        sub-category from the provided list of options.

        Here is data for you to process:
        ---
        article: {article}
        ---
        {format_instructions}
        """,
      input_variables=["article"],
      partial_variables={"format_instructions":subparser.get_format_instructions()}
   )

   sublookup = {v: k for k, v in subcats.items()}
   post_process = RetrieveCode(sublookup, "subcategory", "code")
   subchain = subprompt | model | subparser | post_process
   return subchain



In [ ]:
def categorise_article(input_text, max_depth=2):
    chain = get_root_chain()
    response = chain.invoke({"article":input_text})
    code = response['code']
    code_len = len(code)
    result = {}
    name = "lvl"+str(code_len)
    result[name] = code
    if code != "00":
       while code_len <= max_depth:
          subchain = get_chain(code)
          response2 = subchain.invoke({"article":input_text})
          code = response2['code']
          code_len = len(code)
          name = "lvl"+str(code_len)
          result[name] = code
    return result

In [ ]:
response = categorise_article(article)

print(response)

# Final Script

The ideas in this notebook were compiled into a command line script used to score the sample data sets and evaluate the system performance. See the raw python files:

* [process_json_files.py](process_json_files.py)
* [naics.py](naics.py)